<a href="https://colab.research.google.com/github/PotharaboinaPrasad/BuildMuscleOnline/blob/main/prodigy_infotech_task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets kaggle


In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments
from datasets import load_dataset, Dataset
import pandas as pd



In [ ]:
from google.colab import files
files.upload()  # Upload kaggle.json
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


Saving kaggle.json to kaggle.json


In [ ]:
!kaggle datasets download -d idevji1/sherlock-holmes-stories
!unzip sherlock-holmes-stories.zip -d sherlock_holmes


Dataset URL: https://www.kaggle.com/datasets/idevji1/sherlock-holmes-stories
License(s): CC0-1.0
  0% 0.00/9.93M [00:00<?, ?B/s]
100% 9.93M/9.93M [00:00<00:00, 978MB/s]
Archive:  sherlock-holmes-stories.zip
  inflating: sherlock_holmes/sherlock/3gab.txt  
  inflating: sherlock_holmes/sherlock/3gar.txt  
  inflating: sherlock_holmes/sherlock/3stu.txt  
  inflating: sherlock_holmes/sherlock/abbe.txt  
  inflating: sherlock_holmes/sherlock/advs.txt  
  inflating: sherlock_holmes/sherlock/bery.txt  
  inflating: sherlock_holmes/sherlock/blac.txt  
  inflating: sherlock_holmes/sherlock/blan.txt  
  inflating: sherlock_holmes/sherlock/blue.txt  
  inflating: sherlock_holmes/sherlock/bosc.txt  
  inflating: sherlock_holmes/sherlock/bruc.txt  
  inflating: sherlock_holmes/sherlock/cano.txt  
  inflating: sherlock_holmes/sherlock/card.txt  
  inflating: sherlock_holmes/sherlock/case.txt  
  inflating: sherlock_holmes/sherlock/chas.txt  
  inflating: sherlock_holmes/sherlock/cnus.txt  
  inflati

In [ ]:
import glob
import pandas as pd
from datasets import Dataset

texts = []
for file in glob.glob('/content/sherlock_holmes/sherlock/*.txt'):  # Adjust this path if needed
    with open(file, 'r', encoding='utf-8') as f:
        texts.append(f.read())

print(f"Number of text files loaded: {len(texts)}")

df = pd.DataFrame({'text': texts})
dataset = Dataset.from_pandas(df)

print(f"Loaded dataset size: {len(dataset)}")

Number of text files loaded: 67
Loaded dataset size: 67


In [ ]:
model_name = "gpt2"
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()  # 👈 Needed for loss calculation
    return tokens

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])


Map:   0%|          | 0/67 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    prediction_loss_only=True,
    fp16=True,
    report_to="none" # Disable Weights & Biases logging
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)


In [ ]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


TrainOutput(global_step=102, training_loss=2.2675078148935355, metrics={'train_runtime': 45.7176, 'train_samples_per_second': 4.397, 'train_steps_per_second': 2.231, 'total_flos': 52519698432000.0, 'train_loss': 2.2675078148935355, 'epoch': 3.0})

In [ ]:
# When tokenizing, keep only the columns needed for one step
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Set remove_unused_columns=False in TrainingArguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    prediction_loss_only=True,
    fp16=True,
    remove_unused_columns=False,  # Add this line
    report_to="none" # Disable Weights & Biases logging
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

Map:   0%|          | 0/67 [00:00<?, ? examples/s]

In [ ]:
print("Original dataset size:", len(dataset))
print("Tokenized dataset size:", len(tokenized_dataset))


Original dataset size: 67
Tokenized dataset size: 67


In [ ]:
prompt = "Sherlock Holmes entered the room and"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device) # Move input_ids to the same device as the model
output = model.generate(input_ids, max_length=100)
print(tokenizer.decode(output[0], skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Sherlock Holmes entered the room and sat down on the sofa.
"I'm sorry, Sherlock Holmes," heaped.
"I'm sorry, Sherlock Holmes," I said.
"You know, Sherlock Holmes," he said, "that I'm not entirely sure that you can be so kind as to make me feel so sorry for you."
"I know," I said, "that you are very much in my opinion the most important person in the world. I am sure that
